In [ ]:
# If required, uncomment and run:
# %pip install pandas numpy scikit-learn matplotlib seaborn joblib

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

RANDOM_STATE = 42
TARGET = "Condition_Class"
CLASS_NAMES = ["Excellent", "Good", "Fair", "Poor"]

In [ ]:
# Keep these three CSV files in the same folder as this notebook.
TRAIN_PATH = "../training_image_features.csv"
VAL_PATH   = "../validation_image_features.csv"
TEST_PATH  = "../testing_image_features.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Testing:", test_df.shape)

for name, data in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"\\n{name} class distribution:")
    print(data[TARGET].value_counts())

In [ ]:
EXCLUDE_COLUMNS = [
    TARGET,
    "Sample_ID", "Image_ID",
    "Damage_Score", "Condition_Score",
    "Structural_Integrity", "Recovery_Pathway",
    "Reconditioning_Required", "Recovery_Potential",
    "Alternative_Use_Category", "Alternative_Use_Suitability",
    "pixel_img_width", "pixel_img_height"
]

# Use only columns available in the training data.
feature_columns = [c for c in train_df.columns if c not in EXCLUDE_COLUMNS]

X_train = train_df[feature_columns].copy()
y_train = train_df[TARGET].astype(str)

X_val = val_df[feature_columns].copy()
y_val = val_df[TARGET].astype(str)

X_test = test_df[feature_columns].copy()
y_test = test_df[TARGET].astype(str)

numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [c for c in feature_columns if c not in numeric_features]

print("Number of features:", len(feature_columns))
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("\\nFeatures used:")
print(feature_columns)

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [ ]:
depth_options = [3, 5, 8, 12, 16, 20, None]
validation_results = []
candidate_pipelines = {}

for depth in depth_options:
    cart_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(
            criterion="gini",
            max_depth=depth,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ])

    cart_pipeline.fit(X_train, y_train)
    val_predictions = cart_pipeline.predict(X_val)

    macro_f1 = f1_score(y_val, val_predictions, average="macro", zero_division=0)
    accuracy = accuracy_score(y_val, val_predictions)

    validation_results.append({
        "max_depth": str(depth),
        "validation_accuracy": accuracy,
        "validation_macro_f1": macro_f1
    })
    candidate_pipelines[depth] = cart_pipeline

validation_results_df = pd.DataFrame(validation_results).sort_values(
    "validation_macro_f1", ascending=False
)
display(validation_results_df)

best_depth_label = validation_results_df.iloc[0]["max_depth"]
best_depth = None if best_depth_label == "None" else int(best_depth_label)
print("Selected max_depth:", best_depth)

In [ ]:
X_train_val = pd.concat([X_train, X_val], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)

cart_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        criterion="gini",
        max_depth=best_depth,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

cart_model.fit(X_train_val, y_train_val)
print("Final CART model trained.")

In [ ]:
y_test_pred = cart_model.predict(X_test)

print("CART Test Results")
print("-----------------")
print("Accuracy :", round(accuracy_score(y_test, y_test_pred), 4))
print("Precision (macro):", round(precision_score(y_test, y_test_pred, average="macro", zero_division=0), 4))
print("Recall (macro)   :", round(recall_score(y_test, y_test_pred, average="macro", zero_division=0), 4))
print("F1-score (macro) :", round(f1_score(y_test, y_test_pred, average="macro", zero_division=0), 4))

print("\\nClassification Report:")
print(classification_report(
    y_test, y_test_pred,
    labels=CLASS_NAMES,
    zero_division=0
))

In [ ]:
cm = confusion_matrix(y_test, y_test_pred, labels=CLASS_NAMES)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES
)
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("CART – Test Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Retrieve transformed feature names and CART feature importances.
fitted_preprocessor = cart_model.named_steps["preprocessor"]
tree_classifier = cart_model.named_steps["classifier"]

try:
    transformed_names = fitted_preprocessor.get_feature_names_out()
    importances = tree_classifier.feature_importances_

    importance_df = pd.DataFrame({
        "Feature": transformed_names,
        "Importance": importances
    }).sort_values("Importance", ascending=False)

    display(importance_df.head(20))

    top_features = importance_df.head(20).sort_values("Importance")
    plt.figure(figsize=(9, 7))
    plt.barh(top_features["Feature"], top_features["Importance"])
    plt.xlabel("Feature Importance")
    plt.title("Top 20 CART Feature Importances")
    plt.tight_layout()
    plt.show()
except Exception as error:
    print("Could not display feature names:", error)

In [ ]:
os.makedirs("saved_models", exist_ok=True)

MODEL_PATH = os.path.join("saved_models", "codd_cart_pipeline.joblib")
joblib.dump(cart_model, MODEL_PATH)

print("Saved CART pipeline to:", MODEL_PATH)
print("Use this complete pipeline during ensemble integration so preprocessing is preserved.")